## Task 3

In [3]:
# Run this cell first
# Relevant imports
import pandas as pd
import geopandas as gpd
import numpy as np
import psycopg2
import json
from scipy.stats import iqr

# == DB START UP ==
# Load json config
with open('../credentials.json', 'r') as f:
    config = json.load(f)

# Connection
conn = psycopg2.connect(**config)
cur = conn.cursor()

In [5]:
cur.execute("""
    SELECT *
    FROM sydney_inner_west;
""")
rows = cur.fetchall()
colnames = [desc[0] for desc in cur.description]
df = pd.DataFrame(rows, columns=colnames)

print(f"Raw POI rows loaded: {len(df)}")
print(df.head())

Raw POI rows loaded: 2548
   id                       sa2_name poi_id                 poi_name  \
0   1  Concord - Mortlake - Cabarita    849                     None   
1   2  Concord - Mortlake - Cabarita    852                     None   
2   3  Concord - Mortlake - Cabarita   1120      CONCORD GOLF COURSE   
3   4  Concord - Mortlake - Cabarita   1121       PRINCE EDWARD PARK   
4   5  Concord - Mortlake - Cabarita   1122  MASSEY PARK GOLF COURSE   

       poi_category  poi_type  \
0  Place Of Worship         1   
1  Place Of Worship         1   
2       Golf Course         3   
3              Park         3   
4       Golf Course         3   

                                                geom  
0  0101000020E61000008A5A164781E36240C234E47B33EE...  
1  0101000020E6100000E4F9DD2837E3624025F5FA504CEE...  
2  0101000020E61000008D3997581DE36240DF45BCCBD9EC...  
3  0101000020E6100000E42DF0EACEE362408B38CC0829ED...  
4  0101000020E61000000A4EF29C92E36240E584A89734ED...  


## Population Dataset

Population dataset extracted from ABS Regional Population into region_pop.csv for filtering.
The csv contains only SA2s within our selected SA4 regions and their estimated 2025 population count.
SA2s with population below 100 are excluded BEFORE computing any statistics, so they
do not influence μ or σ.

In [7]:
df_pop = pd.read_csv('region_pop.csv', index_col=0)
df_pop.columns = ['sa4_name', 'sa2_name', 'population']
df_pop['population'] = pd.to_numeric(df_pop['population'], errors='coerce')

# Filter SA2s with population <100
df_pop = df_pop[df_pop['sa4_name'] == 'Sydney - Inner West']
total_before = len(df_pop)

df_pop = df_pop[df_pop['population'] >= 100]

print(f"Total SA2s before filter: {total_before}")
print(f"SA2s removed: {total_before - len(df_pop)}")
print(f"Total SA2s after filter: {len(df_pop)}")
print(df_pop)


#Remove POI rows belonging to excluded SA2s
valid_sa2s = set(df_pop['sa2_name'])
df = df[df['sa2_name'].isin(valid_sa2s)].copy()
print(f"POI rows after SA2 filter: {len(df)}")


Total SA2s before filter: 21
SA2s removed: 0
Total SA2s after filter: 21
               sa4_name                          sa2_name  population
54  Sydney - Inner West     Concord - Mortlake - Cabarita       24026
55  Sydney - Inner West            Drummoyne - Rodd Point       19158
56  Sydney - Inner West            Five Dock - Abbotsford       21449
57  Sydney - Inner West  Concord West - North Strathfield       13453
58  Sydney - Inner West                            Rhodes       13094
59  Sydney - Inner West                           Balmain       16124
60  Sydney - Inner West               Lilyfield - Rozelle       14218
61  Sydney - Inner West                   Annandale (NSW)       10004
62  Sydney - Inner West                        Leichhardt       18357
63  Sydney - Inner West      Canterbury (North) - Ashbury       12633
64  Sydney - Inner West            Croydon Park - Enfield       18270
65  Sydney - Inner West           Dulwich Hill - Lewisham       18826
66  Sydney - Inne

In [9]:
# Method-> groupby + agg: produces one labelled row per SA2 in a single step.
 ##Count POIs per SA2: using pandas groupby + agg
df_scores = df.groupby('sa2_name', as_index=False).agg(
    poi_count=('poi_category', 'count')
)

print(f"\nPOI counts per SA2 ({len(df_scores)} SA2s):")
print(df_scores.sort_values('poi_count', ascending=False).to_string(index=False))



POI counts per SA2 (21 SA2s):
                        sa2_name  poi_count
          Drummoyne - Rodd Point        514
                         Balmain        285
             Lilyfield - Rozelle        276
   Concord - Mortlake - Cabarita        254
                 Annandale (NSW)        138
                      Leichhardt        127
          Five Dock - Abbotsford         96
                   Burwood (NSW)         95
         Dulwich Hill - Lewisham         91
        Haberfield - Summer Hill         88
Concord West - North Strathfield         86
          Croydon Park - Enfield         85
                        Homebush         83
    Canterbury (North) - Ashbury         72
               Strathfield South         56
              Strathfield - East         46
                Ashfield - North         41
              Strathfield - West         41
                         Croydon         31
                Ashfield - South         22
                          Rhodes         21


In [18]:
# Basic Z-score + sigmoid
# z = (x - μ) / σ   where μ = sample mean, σ = sample std
# S = 1 / (1 + e^(-z))

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

mu= df_scores['poi_count'].mean()
sd= df_scores['poi_count'].std()

df_scores['z_poi']=(df_scores['poi_count']-mu)/sd
df_scores['score']=df_scores['z_poi'].apply(sigmoid)


print(f"\nBasic statistics across {len(df_scores)} SA2s in Sydney - Inner West:")
print(f"  μ (mean POI count) = {mu:.2f}")
print(f"  σ (std  POI count) = {sd:.2f}")
print(f"\n SA2s by basic score:")
print(
    df_scores.sort_values('score', ascending=False)
             [['sa2_name', 'poi_count', 'z_poi', 'score']]
             .to_string(index=False)
)



Basic statistics across 21 SA2s in Sydney - Inner West:
  μ (mean POI count) = 121.33
  σ (std  POI count) = 119.08

 SA2s by basic score:
                        sa2_name  poi_count     z_poi    score
          Drummoyne - Rodd Point        514  3.297574 0.964346
                         Balmain        285  1.374456 0.798099
             Lilyfield - Rozelle        276  1.298875 0.785646
   Concord - Mortlake - Cabarita        254  1.114121 0.752897
                 Annandale (NSW)        138  0.139965 0.534934
                      Leichhardt        127  0.047588 0.511895
          Five Dock - Abbotsford         96 -0.212747 0.447013
                   Burwood (NSW)         95 -0.221145 0.444938
         Dulwich Hill - Lewisham         91 -0.254736 0.436658
        Haberfield - Summer Hill         88 -0.279930 0.430471
Concord West - North Strathfield         86 -0.296726 0.426358
          Croydon Park - Enfield         85 -0.305124 0.424305
                        Homebush         

In [27]:
## Extension 1: Robust alternative scoring using Median and IQR

# The mean is sensitive to outliers — one SA2 with an unusually large POI count can disproportionately inflate μ and distort the z-scores for all other SA2s.
# To reduce this sensitivity, a robust alternative replaces:
#  μ  -> median  (50th percentile, less affected by extreme values)
#  σ  -> IQR     (interquartile range = Q3 − Q1, robust spread measure)
# A robust z-score is then calculated using median and IQR then passed through the same sigmoid function, allowing comparison with the basic model.

mu_robust = df_scores['poi_count'].median()
sd_robust = iqr(df_scores['poi_count'])

df_scores['z_poi_robust']  = (df_scores['poi_count'] - mu_robust) / sd_robust
df_scores['score_robust']  = df_scores['z_poi_robust'].apply(sigmoid)

print(f"Robust statistics:")
print(f"  median = {mu_robust:.2f}  (vs mean = {mu:.2f})")
print(f"  IQR    = {sd_robust:.2f}  (vs std  = {sd:.2f})")

print(f"\nTop 5 SA2s by BASIC score:")
print(
    df_scores.sort_values('score', ascending=False)
             .head(5)[['sa2_name', 'poi_count', 'score']]
             .to_string(index=False)
)
print(f"\nTop 5 SA2s by ROBUST score (median/IQR):")
print(
    df_scores.sort_values('score_robust', ascending=False)
             .head(5)[['sa2_name', 'poi_count', 'score_robust']]
             .to_string(index=False)
)


Robust statistics:
  median = 86.00  (vs mean = 121.33)
  IQR    = 81.00  (vs std  = 119.08)

Top 5 SA2s by BASIC score:
                     sa2_name  poi_count    score
       Drummoyne - Rodd Point        514 0.964346
                      Balmain        285 0.798099
          Lilyfield - Rozelle        276 0.785646
Concord - Mortlake - Cabarita        254 0.752897
              Annandale (NSW)        138 0.534934

Top 5 SA2s by ROBUST score (median/IQR):
                     sa2_name  poi_count  score_robust
       Drummoyne - Rodd Point        514      0.994953
                      Balmain        285      0.921057
          Lilyfield - Rozelle        276      0.912590
Concord - Mortlake - Cabarita        254      0.888358
              Annandale (NSW)        138      0.655200


In [22]:
##Extension 2- weighted POI types
##JUSTIFICATION:
# The basic score treats every POI equally: a General Hospital and a Quarry each contribute 1.  This undervalues essential services and overvalues POIs with low community utility.
# We replace the raw count with a quality-adjusted weighted count.
# Weights are assigned at two levels:
#   (a) poi_type  — broad category multiplier (e.g. Education = 2.0)
#   (b) poi_category — fine-grained override for specific POI names;
#       the category override takes priority over the type-level weight.
#
# The same z-score + sigmoid formula is then applied to weighted_poi, so the only change is that it is now quality-adjusted.

type_weights = {
    1: 1.5, #Community
    2: 2.0, #Education
    3: 1.5, #Recreation
    4: 1.5  #Transport
}

DEFAULT_TYPE_WEIGHT = 1.0

category_overrides = {
    #Health
    'General Hospital': 3.0,
    'Community Medical Centre': 3.0,

    #Emergency services
    'Police Station': 2.5,
    'Fire Station': 2.5,
    'Firestation - Bush': 2.0,
    'SES Facility': 2.0,

    #Civic
    'Library': 2.0,
    'Post Office': 1.5,
    'Shopping Center': 2.0,

    #Low value
    'Quarry - Open Cut':0.1,
    'Dam Wall': 0.1,
    'Zoo': 0.2,
    'Cemetery': 0.2,
    'Motor Racing Track': 0.2,
    'Rubbish Depot': 0.2
}

def get_weight(row):
    if row['poi_category'] in category_overrides:
        return category_overrides[row['poi_category']]
    return type_weights.get(int(row['poi_type']), DEFAULT_TYPE_WEIGHT)

df['weight'] = df.apply(get_weight, axis = 1)

df_weighted = df.groupby('sa2_name', as_index=False)['weight'].sum()
df_weighted.columns = ['sa2_name', 'weighted_poi']

df_scores = df_scores.merge(df_weighted, on = 'sa2_name', how = 'left')


In [25]:
# Weighted Z-score
mu_w= df_scores['weighted_poi'].mean()
sd_w= df_scores['weighted_poi'].std()

df_scores['z_poi_weighted'] = (df_scores['weighted_poi'] - mu_w) /sd_w
df_scores['weighted_score'] = df_scores['z_poi_weighted'].apply(sigmoid)

print(f"\nWeighted statistics across {len(df_scores)} SA2s:")
print(f"  μ_w = {mu_w:.2f},  σ_w = {sd_w:.2f}")
print(f"\nTop 5 SA2s by weighted score:")
print(
    df_scores.sort_values('weighted_score', ascending=False)
             .head(5)[['sa2_name', 'weighted_poi', 'z_poi_weighted', 'weighted_score']]
             .to_string(index=False)
)


Weighted statistics across 21 SA2s:
  μ_w = 185.39,  σ_w = 176.75

Top 5 SA2s by weighted score:
                     sa2_name  weighted_poi  z_poi_weighted  weighted_score
       Drummoyne - Rodd Point         765.0        3.279263        0.963711
                      Balmain         430.0        1.383945        0.799624
          Lilyfield - Rozelle         413.5        1.290594        0.784248
Concord - Mortlake - Cabarita         382.0        1.112378        0.752572
              Annandale (NSW)         208.2        0.129076        0.532224


In [26]:
df_scores = df_scores.merge(
    df_pop[['sa2_name', 'population']], on='sa2_name', how='inner'
)

df_final = (
    df_scores
    .sort_values('weighted_score', ascending=False)
    .reset_index(drop=True)
)
df_final.index += 1   # rank starts at 1

print("\n── Final SA2 Rankings — Sydney - Inner West")
print(df_final[[
    'sa2_name', 'population',
    'poi_count', 'score',
    'weighted_poi', 'weighted_score'
]].to_string())



── Final SA2 Rankings — Sydney - Inner West
                            sa2_name  population  poi_count     score  weighted_poi  weighted_score
1             Drummoyne - Rodd Point       19158        514  0.964346         765.0        0.963711
2                            Balmain       16124        285  0.798099         430.0        0.799624
3                Lilyfield - Rozelle       14218        276  0.785646         413.5        0.784248
4      Concord - Mortlake - Cabarita       24026        254  0.752897         382.0        0.752572
5                    Annandale (NSW)       10004        138  0.534934         208.2        0.532224
6                         Leichhardt       18357        127  0.511895         197.0        0.516422
7                      Burwood (NSW)       18684         95  0.444938         167.5        0.474724
8             Five Dock - Abbotsford       21449         96  0.447013         148.5        0.448017
9            Dulwich Hill - Lewisham       18826       

In [10]:
cur.execute("""
    CREATE TABLE IF NOT EXISTS sa2_scores (
        sa2_name            TEXT PRIMARY KEY,
        sa4_zone            TEXT,
        poi_count           INTEGER,
        population          INTEGER,
        z_poi               NUMERIC,
        score               NUMERIC,
        weighted_poi        NUMERIC,
        z_poi_weighted      NUMERIC,
        weighted_score      NUMERIC
    );
""")
cur.execute("TRUNCATE TABLE sa2_scores RESTART IDENTITY;")
conn.commit()

rows = [
    (
        row['sa2_name'],
        'Sydney - Inner West',
        int(row['poi_count']),
        int(row['population']),
        float(row['z_poi']),
        float(row['score']),
        float(row['weighted_poi']),
        float(row['z_poi_weighted']),
        float(row['weighted_score'])
    )
    for _, row in df_scores.iterrows()
]

cur.executemany("""
    INSERT INTO sa2_scores
        (sa2_name, sa4_zone, poi_count, population, z_poi,
         score, weighted_poi, z_poi_weighted, weighted_score)
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
    ON CONFLICT (sa2_name) DO UPDATE
        SET poi_count          = EXCLUDED.poi_count,
            population         = EXCLUDED.population,
            z_poi              = EXCLUDED.z_poi,
            score              = EXCLUDED.score,
            weighted_poi       = EXCLUDED.weighted_poi,
            z_poi_weighted     = EXCLUDED.z_poi_weighted,
            weighted_score     = EXCLUDED.weighted_score
""", rows)
conn.commit()

print(f"Scores stored for {len(rows)} SA2s.")


Scores stored for 21 SA2s.


In [11]:
cur.close()
conn.close()